#**Importing Spark Resources**

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import * 
spark = SparkSession.builder.appName('Project-1').getOrCreate()

#Reading Files From Upcoming Source

In [0]:
df = spark.read.format("csv")\
        .option("header","true")\
            .option('inferschema','true')\
                .load('/Volumes/project1/default/data/customer.csv')

#Creating Schema inside Catalog

In [0]:
%sql
create schema if not exists project1.src;

#Writing Table in Delta Format

In [0]:
df.write.format('delta')\
    .mode('overwrite')\
        .saveAsTable('project1.src.customer')

#Display

In [0]:
%sql
select * from src.customer

customer_id,first_name,last_name,email,city,state,signup_date,is_active
1,Alice,Johnson,alice.johnson@email.com,New York,NY,2024-01-15,true
2,Bob,Smith,bob.smith@email.com,Chicago,IL,2024-02-20,true
3,Charlie,Brown,charlie.brown@email.com,Dallas,TX,2024-03-05,true
4,Diana,Prince,diana.prince@email.com,Los Angeles,CA,2024-03-18,true
5,Ethan,Hunt,ethan.hunt@email.com,Seattle,WA,2024-04-10,false
6,Fiona,Clark,fiona.clark@email.com,Miami,FL,2024-05-22,true
7,George,King,george.king@email.com,Boston,MA,2024-06-01,true
8,Hannah,Scott,hannah.scott@email.com,Denver,CO,2024-06-15,false
9,Ian,Wright,ian.wright@email.com,Atlanta,GA,2024-07-03,true
10,Julia,Roberts,julia.roberts@email.com,Phoenix,AZ,2024-07-20,true


#Adding Columns

In [0]:
spark.sql("""
          alter table project1.src.customer 
          add columns (
            full_name varchar(200)
          )
          """)

DataFrame[]

In [0]:
%sql
update src.customer 
set full_name = concat(first_name, ' ', last_name) 
where full_name is null;
    
select * from src.customer

customer_id,first_name,last_name,email,city,state,signup_date,is_active,full_name
1,Alice,Johnson,alice.johnson@email.com,New York,NY,2024-01-15,true,Alice Johnson
2,Bob,Smith,bob.smith@email.com,Chicago,IL,2024-02-20,true,Bob Smith
3,Charlie,Brown,charlie.brown@email.com,Dallas,TX,2024-03-05,true,Charlie Brown
4,Diana,Prince,diana.prince@email.com,Los Angeles,CA,2024-03-18,true,Diana Prince
5,Ethan,Hunt,ethan.hunt@email.com,Seattle,WA,2024-04-10,false,Ethan Hunt
6,Fiona,Clark,fiona.clark@email.com,Miami,FL,2024-05-22,true,Fiona Clark
7,George,King,george.king@email.com,Boston,MA,2024-06-01,true,George King
8,Hannah,Scott,hannah.scott@email.com,Denver,CO,2024-06-15,false,Hannah Scott
9,Ian,Wright,ian.wright@email.com,Atlanta,GA,2024-07-03,true,Ian Wright
10,Julia,Roberts,julia.roberts@email.com,Phoenix,AZ,2024-07-20,true,Julia Roberts


#History 

In [0]:
%sql
describe history src.customer

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
3,2026-03-03T02:10:06.000Z,77489229441512,sviscoding@gmail.com,UPDATE,"Map(predicate -> [""isnull(full_name#15602)""])",null,List(804868019836470),13b0cb89-0034-4a2a-94d3-167ffce9c303,0303-005637-nembbzm-v2n,2,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 827, numDeletionVectorsUpdated -> 0, scanTimeMs -> 826, numAddedFiles -> 0, numUpdatedRows -> 0, numAddedBytes -> 0, rewriteTimeMs -> 0)",null,Databricks-Runtime/18.0.x-aarch64-photon-scala2.13
2,2026-03-03T02:09:48.000Z,77489229441512,sviscoding@gmail.com,UPDATE,Map(predicate -> []),null,List(804868019836470),8c4cf179-2e4d-468b-966b-6cfb7a9de381,0303-005637-nembbzm-v2n,1,WriteSerializable,false,"Map(numRemovedFiles -> 1, numRemovedBytes -> 2630, numCopiedRows -> 0, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 3675, numDeletionVectorsUpdated -> 0, scanTimeMs -> 31, numAddedFiles -> 1, numUpdatedRows -> 10, numAddedBytes -> 3036, rewriteTimeMs -> 3615)",null,Databricks-Runtime/18.0.x-aarch64-photon-scala2.13
1,2026-03-03T02:08:17.000Z,77489229441512,sviscoding@gmail.com,ADD COLUMNS,"Map(columns -> [{""column"":{""name"":""full_name"",""type"":""varchar(200)"",""nullable"":true,""metadata"":{}}}])",null,List(804868019836470),368a8088-db41-4183-b4fe-3c357726f014,0303-005637-nembbzm-v2n,0,WriteSerializable,true,Map(),null,Databricks-Runtime/18.0.x-aarch64-photon-scala2.13
0,2026-03-03T01:38:29.000Z,77489229441512,sviscoding@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(804868019836470),ad7b46c4-5a62-4454-91bb-2d7631c973f6,0303-005637-nembbzm-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 10, numOutputBytes -> 2630)",null,Databricks-Runtime/18.0.x-aarch64-photon-scala2.13


#Time Traver Before Adding Column

In [0]:
%sql
select * from src.customer version as of 0

customer_id,first_name,last_name,email,city,state,signup_date,is_active
1,Alice,Johnson,alice.johnson@email.com,New York,NY,2024-01-15,true
2,Bob,Smith,bob.smith@email.com,Chicago,IL,2024-02-20,true
3,Charlie,Brown,charlie.brown@email.com,Dallas,TX,2024-03-05,true
4,Diana,Prince,diana.prince@email.com,Los Angeles,CA,2024-03-18,true
5,Ethan,Hunt,ethan.hunt@email.com,Seattle,WA,2024-04-10,false
6,Fiona,Clark,fiona.clark@email.com,Miami,FL,2024-05-22,true
7,George,King,george.king@email.com,Boston,MA,2024-06-01,true
8,Hannah,Scott,hannah.scott@email.com,Denver,CO,2024-06-15,false
9,Ian,Wright,ian.wright@email.com,Atlanta,GA,2024-07-03,true
10,Julia,Roberts,julia.roberts@email.com,Phoenix,AZ,2024-07-20,true


#Deleting inactive rows

In [0]:
spark.sql("""
          delete from project1.src.customer
          where is_active = false
          """)

DataFrame[num_affected_rows: bigint]

In [0]:
%sql
select * from src.customer

customer_id,first_name,last_name,email,city,state,signup_date,is_active,full_name
1,Alice,Johnson,alice.johnson@email.com,New York,NY,2024-01-15,true,Alice Johnson
2,Bob,Smith,bob.smith@email.com,Chicago,IL,2024-02-20,true,Bob Smith
3,Charlie,Brown,charlie.brown@email.com,Dallas,TX,2024-03-05,true,Charlie Brown
4,Diana,Prince,diana.prince@email.com,Los Angeles,CA,2024-03-18,true,Diana Prince
6,Fiona,Clark,fiona.clark@email.com,Miami,FL,2024-05-22,true,Fiona Clark
7,George,King,george.king@email.com,Boston,MA,2024-06-01,true,George King
9,Ian,Wright,ian.wright@email.com,Atlanta,GA,2024-07-03,true,Ian Wright
10,Julia,Roberts,julia.roberts@email.com,Phoenix,AZ,2024-07-20,true,Julia Roberts


#Time Travel - Restoring Previous State 

In [0]:
%sql
describe history src.customer

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
5,2026-03-03T02:19:51.000Z,77489229441512,sviscoding@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(804868019836470),b2f14910-79ca-43f0-bb7a-a8c0eb5617e4,0303-005637-nembbzm-v2n,4,SnapshotIsolation,false,"Map(numRemovedFiles -> 1, numRemovedBytes -> 3036, p25FileSize -> 2934, numDeletionVectorsRemoved -> 1, minFileSize -> 2934, numAddedFiles -> 1, maxFileSize -> 2934, p75FileSize -> 2934, p50FileSize -> 2934, numAddedBytes -> 2934)",null,Databricks-Runtime/18.0.x-aarch64-photon-scala2.13
4,2026-03-03T02:19:48.000Z,77489229441512,sviscoding@gmail.com,DELETE,"Map(predicate -> [""NOT is_active#16159""])",null,List(804868019836470),b2f14910-79ca-43f0-bb7a-a8c0eb5617e4,0303-005637-nembbzm-v2n,3,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 2206, numDeletionVectorsUpdated -> 0, numDeletedRows -> 2, scanTimeMs -> 1373, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 815)",null,Databricks-Runtime/18.0.x-aarch64-photon-scala2.13
3,2026-03-03T02:10:06.000Z,77489229441512,sviscoding@gmail.com,UPDATE,"Map(predicate -> [""isnull(full_name#15602)""])",null,List(804868019836470),13b0cb89-0034-4a2a-94d3-167ffce9c303,0303-005637-nembbzm-v2n,2,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 827, numDeletionVectorsUpdated -> 0, scanTimeMs -> 826, numAddedFiles -> 0, numUpdatedRows -> 0, numAddedBytes -> 0, rewriteTimeMs -> 0)",null,Databricks-Runtime/18.0.x-aarch64-photon-scala2.13
2,2026-03-03T02:09:48.000Z,77489229441512,sviscoding@gmail.com,UPDATE,Map(predicate -> []),null,List(804868019836470),8c4cf179-2e4d-468b-966b-6cfb7a9de381,0303-005637-nembbzm-v2n,1,WriteSerializable,false,"Map(numRemovedFiles -> 1, numRemovedBytes -> 2630, numCopiedRows -> 0, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 3675, numDeletionVectorsUpdated -> 0, scanTimeMs -> 31, numAddedFiles -> 1, numUpdatedRows -> 10, numAddedBytes -> 3036, rewriteTimeMs -> 3615)",null,Databricks-Runtime/18.0.x-aarch64-photon-scala2.13
1,2026-03-03T02:08:17.000Z,77489229441512,sviscoding@gmail.com,ADD COLUMNS,"Map(columns -> [{""column"":{""name"":""full_name"",""type"":""varchar(200)"",""nullable"":true,""metadata"":{}}}])",null,List(804868019836470),368a8088-db41-4183-b4fe-3c357726f014,0303-005637-nembbzm-v2n,0,WriteSerializable,true,Map(),null,Databricks-Runtime/18.0.x-aarch64-photon-scala2.13
0,2026-03-03T01:38:29.000Z,77489229441512,sviscoding@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(804868019836470),ad7b46c4-5a62-4454-91bb-2d7631c973f6,0303-005637-nembbzm-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 10, numOutputBytes -> 2630)",null,Databricks-Runtime/18.0.x-aarch64-photon-scala2.13


In [0]:
%sql
restore table project1.src.customer to version as of 3

table_size_after_restore,num_of_files_after_restore,num_removed_files,num_restored_files,removed_files_size,restored_files_size
3036,1,1,1,2934,3036


In [0]:
%sql
select * from project1.src.customer

customer_id,first_name,last_name,email,city,state,signup_date,is_active,full_name
1,Alice,Johnson,alice.johnson@email.com,New York,NY,2024-01-15,true,Alice Johnson
2,Bob,Smith,bob.smith@email.com,Chicago,IL,2024-02-20,true,Bob Smith
3,Charlie,Brown,charlie.brown@email.com,Dallas,TX,2024-03-05,true,Charlie Brown
4,Diana,Prince,diana.prince@email.com,Los Angeles,CA,2024-03-18,true,Diana Prince
5,Ethan,Hunt,ethan.hunt@email.com,Seattle,WA,2024-04-10,false,Ethan Hunt
6,Fiona,Clark,fiona.clark@email.com,Miami,FL,2024-05-22,true,Fiona Clark
7,George,King,george.king@email.com,Boston,MA,2024-06-01,true,George King
8,Hannah,Scott,hannah.scott@email.com,Denver,CO,2024-06-15,false,Hannah Scott
9,Ian,Wright,ian.wright@email.com,Atlanta,GA,2024-07-03,true,Ian Wright
10,Julia,Roberts,julia.roberts@email.com,Phoenix,AZ,2024-07-20,true,Julia Roberts


#Liquid Clustering

In [0]:
%sql
alter table project1.src.customer 
cluster by (city,state)

#Optimizing

In [0]:
%sql 
OPTIMIZE project1.src.customer


path,metrics
,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 1, 0, false, 0, 0, 1772505165734, 1772505171772, 8, 0, null, List(0, 0), null, 9, 9, 0, 0, List(3036, true, false, false, null, null, null, null, 0, 0, 0, 0, 1, 3036, 3036, null, log, 16777216, 67108864, 4, 0, 0, null, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, List(189, 106, 0, 0, 0, 2436), 2, 1, 5, sizeAware, false, 0, null), null)"
,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 1, 1, true, 0, 0, 1772505171822, 1772505174102, 8, 0, null, List(0, 0), null, 9, 9, 0, 0, List(3036, false, false, false, null, null, null, post-optimize-compaction, 0, 0, 0, 0, 0, 0, 0, null, null, 33554432, 67108864, 0, 0, 0, null, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, List(0, 0, 1183, 0, 0, 0), 15, 1, 1, null, false, 0, null), null)"


#Vaccum

In [0]:
%sql
vacuum project1.src.customer 

path
""


In [0]:
%sql
describe history project1.src.customer

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
12,2026-03-03T02:36:10.000Z,77489229441512,sviscoding@gmail.com,VACUUM END,Map(status -> COMPLETED),null,List(804868019836470),86b63280-7f88-44f7-9e22-f06d1b416aa1,0303-005637-nembbzm-v2n,11,SnapshotIsolation,true,"Map(numDeletedFiles -> 0, numVacuumedDirectories -> 1)",null,Databricks-Runtime/18.0.x-aarch64-photon-scala2.13
11,2026-03-03T02:36:08.000Z,77489229441512,sviscoding@gmail.com,VACUUM START,"Map(retentionCheckEnabled -> true, defaultRetentionMillis -> 604800000)",null,List(804868019836470),86b63280-7f88-44f7-9e22-f06d1b416aa1,0303-005637-nembbzm-v2n,10,SnapshotIsolation,true,"Map(numFilesToDelete -> 0, sizeOfDataToDelete -> 0)",null,Databricks-Runtime/18.0.x-aarch64-photon-scala2.13
10,2026-03-03T02:32:51.000Z,77489229441512,sviscoding@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> false, clusterBy -> [""city"",""state""], isFull -> false, zOrderBy -> [], batchId -> -1)",null,List(804868019836470),50fbe41a-7c7a-49e2-a2d5-a56bba92bb5d,0303-005637-nembbzm-v2n,9,SnapshotIsolation,true,Map(),null,Databricks-Runtime/18.0.x-aarch64-photon-scala2.13
9,2026-03-03T02:31:11.000Z,77489229441512,sviscoding@gmail.com,CLUSTER BY,"Map(oldClusteringColumns -> , newClusteringColumns -> city,state)",null,List(804868019836470),bdb5ecdd-fe3e-46db-b1b4-1fce35c432ff,0303-005637-nembbzm-v2n,8,WriteSerializable,true,Map(),null,Databricks-Runtime/18.0.x-aarch64-photon-scala2.13
8,2026-03-03T02:31:10.000Z,77489229441512,sviscoding@gmail.com,ROW TRACKING BACKFILL,Map(batchId -> 0),null,List(804868019836470),bdb5ecdd-fe3e-46db-b1b4-1fce35c432ff,0303-005637-nembbzm-v2n,7,SnapshotIsolation,false,Map(),null,Databricks-Runtime/18.0.x-aarch64-photon-scala2.13
7,2026-03-03T02:31:08.000Z,77489229441512,sviscoding@gmail.com,UPGRADE PROTOCOL,"Map(newProtocol -> {""minReaderVersion"":3,""minWriterVersion"":7,""readerFeatures"":[""deletionVectors""],""writerFeatures"":[""deletionVectors"",""domainMetadata"",""rowTracking"",""invariants"",""appendOnly""]})",null,List(804868019836470),bdb5ecdd-fe3e-46db-b1b4-1fce35c432ff,0303-005637-nembbzm-v2n,6,WriteSerializable,true,Map(),null,Databricks-Runtime/18.0.x-aarch64-photon-scala2.13
6,2026-03-03T02:23:52.000Z,77489229441512,sviscoding@gmail.com,RESTORE,"Map(version -> 3, timestamp -> null)",null,List(804868019836470),d8317115-e31e-4e23-8e7f-c1462c190148,0303-005637-nembbzm-v2n,5,Serializable,false,"Map(numRestoredFiles -> 1, removedFilesSize -> 2934, numRemovedFiles -> 1, restoredFilesSize -> 3036, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 0, numOfFilesAfterRestore -> 1, tableSizeAfterRestore -> 3036)",null,Databricks-Runtime/18.0.x-aarch64-photon-scala2.13
5,2026-03-03T02:19:51.000Z,77489229441512,sviscoding@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(804868019836470),b2f14910-79ca-43f0-bb7a-a8c0eb5617e4,0303-005637-nembbzm-v2n,4,SnapshotIsolation,false,"Map(numRemovedFiles -> 1, numRemovedBytes -> 3036, p25FileSize -> 2934, numDeletionVectorsRemoved -> 1, minFileSize -> 2934, numAddedFiles -> 1, maxFileSize -> 2934, p75FileSize -> 2934, p50FileSize -> 2934, numAddedBytes -> 2934)",null,Databricks-Runtime/18.0.x-aarch64-photon-scala2.13
4,2026-03-03T02:19:48.000Z,77489229441512,sviscoding@gmail.com,DELETE,"Map(predicate -> [""NOT is_active#16159""])",null,List(804868019836470),b2f14910-79ca-43f0-bb7a-a8c0eb5617e4,0303-005637-nembbzm-v2n,3,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 2206, numDeletionVectorsUpdated -> 0, numDeletedRows -> 2, scanTimeMs -> 1373, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 815)",null,Databricks-Runtime/18.0.x-aarch64-photon-scala2.13
3,20